# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split,cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import urllib.request
import seaborn as sns
import matplotlib.pyplot as plt
import toolbox_ML_v2 as tl
import re
import bootcampviztools as bt


from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from transformer import (
    transformer_screenresolution,
    transformer_cpu,
    transformer_memory,
    transformer_gpu,
    transformer_weight,
    transformer_ram,
)

## 2. Datos

In [2]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv", index_col = 'laptop_ID')

### 2.1 Exploración de los datos

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   ScreenResolution  912 non-null    object 
 5   Cpu               912 non-null    object 
 6   Ram               912 non-null    object 
 7   Memory            912 non-null    object 
 8   Gpu               912 non-null    object 
 9   OpSys             912 non-null    object 
 10  Weight            912 non-null    object 
 11  Price_in_euros    912 non-null    float64
dtypes: float64(2), object(10)
memory usage: 92.6+ KB


In [164]:
# df.head()

In [165]:
# df.tail()

In [166]:
# df.describe()

### 2.3 Definir X e y

In [4]:
X = df.drop(['Price_in_euros', "Product"], axis=1)
y = df['Price_in_euros'].copy()
X.shape

(912, 10)

In [168]:
y.shape

(912,)

### 2.4 Dividir X_train, X_test, y_train, y_test

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

In [6]:
X_train.shape

(729, 10)

In [7]:
X_test.shape

(183, 10)

In [8]:
y_train.shape

(729,)

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

In [9]:
target = 'Price_in_euros'

#### Company, TypeName y OpSys (One-Hot Encoding)

In [10]:
one_hot = ['Company','TypeName','OpSys']

In [11]:
# for col in one_hot:
#     print (f'{col}= {X_train[col].nunique()}')
#     print(X_train[col].value_counts())
#     print('\n')

A Company, TypeName y OpSys les hago One-Hot Encoding porque son datos tipo “etiqueta” (marca, tipo de laptop y sistema operativo). No tienen un orden numérico real. Con one-hot, cada opción se transforma en una columna 0/1 y el modelo puede aprender su efecto sin confundirse. 

#### Columna Product

In [12]:
# X_train['Product'].sample(5)

Esta columna tiene muchísimos valores distintos (cardinalidad muy alta), lo que haría que un one-hot genere demasiadas columnas y ruido. Además, suele funcionar más como un identificador/etiqueta específica que como una señal general. Por eso la eliminamos del dataset antes de entrenar.

#### Columna Inches

In [13]:
X_train['Inches'].sample(5)

laptop_ID
974     13.3
16      13.3
171     15.6
1103    13.3
196     13.3
Name: Inches, dtype: float64

Esta columna ya viene en formato numérico y representa el tamaño de pantalla en pulgadas. Como está “limpia” y es una variable útil, no necesita transformación y se deja tal cual para el modelo.

#### Columna ScreenResolution

In [14]:
X_train['ScreenResolution'].sample(5)

laptop_ID
655    Full HD 1920x1080
305             1366x768
910             1366x768
788            1920x1080
725             1366x768
Name: ScreenResolution, dtype: object

En esta columna tenemos un string que combina información sobre la pantalla. En la mayoría de las filas aparece la resolución en píxeles con el formato ancho x alto (por ejemplo, 1920x1080). Esta la vamos a conservar.

Además, en algunas filas se indica si la pantalla es táctil (“Touchscreen”). Esto también puede aportar señal al modelo, porque las pantallas táctiles suelen asociarse a determinados segmentos de producto.

Qué vamos a extraer (features):
- Res_X: ancho de la resolución en píxeles (ej. 1920)
- Res_Y: alto de la resolución en píxeles (ej. 1080)
- Touchscreen: variable binaria (0/1) que indica si el texto contiene “Touchscreen”

Luego podemos afinar con:

Podemos agregar una variable ordinal de “calidad de pantalla” (por ejemplo, HD < Full HD < Ultra HD/4K) 

#### Columna Gpu

In [15]:
X_train['Gpu'].sample(5)

laptop_ID
496     Intel HD Graphics 620
869     Intel HD Graphics 520
434     Intel HD Graphics 620
1218       AMD Radeon R7 M440
979        AMD Radeon R5 M430
Name: Gpu, dtype: object

En la columna Gpu la información viene como texto (marca + familia +, a veces, número de modelo). Para que el modelo pueda usarla, la convertimos en variables estructuradas: identificamos la marca (Intel/Nvidia/AMD/otras), marcamos si pertenece a una familia (GTX/RTX/MX/Quadro/RX/FirePro), diferenciamos dedicada vs integrada, y cuando aparece, extraemos el número de modelo (por ejemplo 1050, 620), guardándolo por familia. Mantener “flag + número” evita perder información cuando la familia está presente pero el número no se puede extraer.

#### Columna Cpu

In [180]:
X_train['Cpu'].sample(5)

laptop_ID
731    Intel Core i7 7700HQ 2.8GHz
657     Intel Core i7 7500U 2.7GHz
228     Intel Core i5 8250U 1.6GHz
475    Intel Core i7 7700HQ 2.8GHz
610    Intel Core i7 6820HK 2.7GHz
Name: Cpu, dtype: object

En la columna Cpu el dato viene como texto (por ejemplo: “Intel Core i7… 2.8GHz”). Para usarlo en el modelo lo pasamos a columnas más simples: marcamos la marca (Intel/AMD/otra), detectamos la familia (Core i, Core M, Ryzen, A-Series o Celeron/Pentium/Atom) y, si aparece, sacamos la velocidad en GHz. Con eso el modelo puede comparar CPUs sin leer texto.   
Gama dentro de la familia: cuando aparece, extraemos un valor ordinal para Core i3/i5/i7/i9, Core M, Ryzen y A-Series (por ejemplo, i7 → 7)

#### Columna Memory

In [181]:
X_train['Memory'].sample(5)

laptop_ID
7       256GB Flash Storage
324     128GB Flash Storage
955                 1TB HDD
855               256GB SSD
1186     32GB Flash Storage
Name: Memory, dtype: object

En la columna Memory el dato viene como texto y a veces trae una o dos partes (por ejemplo: “256GB SSD + 1TB HDD”). Para que el modelo lo entienda, separamos cada parte, detectamos el tipo de almacenamiento (SSD, HDD, Flash Storage o Hybrid) y sumamos la capacidad correspondiente. Finalmente convertimos todo a una misma unidad (GB, pasando TB → GB) y dejamos cuatro columnas numéricas: SSD_GB, HDD_GB, Flash_Storage_GB y Hybrid_GB.

#### Columnas Ram y Weight

In [182]:
X_train['Ram'].sample(5)

laptop_ID
57       4GB
285      6GB
137      8GB
1081    64GB
309      8GB
Name: Ram, dtype: object

In [183]:
X_train['Weight'].sample(5)

laptop_ID
613      2.2kg
281      2.8kg
711      2.1kg
26       2.3kg
1252    1.08kg
Name: Weight, dtype: object

En Ram y Weight los valores venían como texto con la unidad. Lo que hicimos fue quitar la unidad y convertirlos a número, creando dos columnas limpias: Ram_GB (entero) y Weight_kg (float).

### Pipeline

In [16]:
def wrap_screen(X: pd.DataFrame):
    col = X.columns[0]
    return transformer_screenresolution(X, col)

def wrap_cpu(X: pd.DataFrame):
    col = X.columns[0]
    return transformer_cpu(X, col)

def wrap_memory(X: pd.DataFrame):
    col = X.columns[0]
    return transformer_memory(X, col)

def wrap_gpu(X: pd.DataFrame):
    col = X.columns[0]
    return transformer_gpu(X, col)

def wrap_weight(X: pd.DataFrame):
    col = X.columns[0]
    return transformer_weight(X, col)

def wrap_ram(X: pd.DataFrame):
    col = X.columns[0]
    return transformer_ram(X, col)

ft_screen = FunctionTransformer(wrap_screen, validate=False)
ft_cpu = FunctionTransformer(wrap_cpu, validate=False)
ft_memory = FunctionTransformer(wrap_memory, validate=False)
ft_gpu = FunctionTransformer(wrap_gpu, validate=False)
ft_weight = FunctionTransformer(wrap_weight, validate=False)
ft_ram = FunctionTransformer(wrap_ram, validate=False)



In [17]:
categorical_cols = ["Company", "TypeName", "OpSys"]
numeric_passthrough = ["Inches"]  

preprocessor = ColumnTransformer(
    transformers=[
        ("screen", ft_screen, ["ScreenResolution"]),
        ("cpu", ft_cpu, ["Cpu"]),
        ("memory", ft_memory, ["Memory"]),
        ("gpu", ft_gpu, ["Gpu"]),
        ("weight", ft_weight, ["Weight"]),
        ("ram", ft_ram, ["Ram"]),
        ("cat_ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("inches", "passthrough", numeric_passthrough),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)


In [18]:
preprocessor.set_output(transform="pandas")

X_train_clean = preprocessor.fit_transform(X_train)
X_test_clean = preprocessor.transform(X_test)

print(type(X_train_clean))
print(X_train_clean.shape)
X_train_clean.head()


<class 'pandas.core.frame.DataFrame'>
(729, 75)


,Res_X,Res_Y,Touchscreen,CPU_Intel,CPU_AMD,CPU_Other,CPU_Core_Ord,CPU_CoreM_Ord,CPU_Ryzen_Ord,CPU_ASeries_Ord,...,OpSys_Android,OpSys_Chrome OS,OpSys_Linux,OpSys_Mac OS X,OpSys_No OS,OpSys_Windows 10,OpSys_Windows 10 S,OpSys_Windows 7,OpSys_macOS,Inches
laptop_ID,,,,,,,,,,,,,,,,,,,,,
1118,1920,1080,0,1,0,0,7,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,17.3
153,1920,1080,0,1,0,0,7,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,15.6
275,2560,1600,0,1,0,0,5,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13.3
1100,1920,1080,0,1,0,0,5,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,14.0
131,1920,1080,0,1,0,0,7,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,17.3


-----------------------------------------------------------------------------------------------------------------

## 4. Modelado

### 4.1 Baseline de modelos


In [19]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

models = {
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "XGBRegressor": XGBRegressor(
        random_state=42,
        objective="reg:squarederror",
    ),
    "LGBMRegressor": LGBMRegressor(random_state=42),
    "CatBoostRegressor": CatBoostRegressor(
        random_state=42,
        loss_function="RMSE",
        verbose=False,
    ),
}


In [20]:

results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    mse_scores = -scores                 # MSE por fold
    rmse_scores = np.sqrt(mse_scores)    # RMSE por fold
    results.append({
        "model": name,
        "mean_RMSE": rmse_scores.mean(),  # solo RMSE medio
    })

results_df = pd.DataFrame(results).sort_values("mean_RMSE")
print(results_df)



               model   mean_RMSE
4  CatBoostRegressor  255.950120
1       RandomForest  280.714677
2       XGBRegressor  287.519712
3      LGBMRegressor  292.128036
0       DecisionTree  376.238144


### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

In [21]:
best_model_name = results_df.iloc[0]["model"]
print("Mejor modelo básico:", best_model_name)

# recreamos el pipeline con ese modelo
best_base_model = models[best_model_name]
best_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", best_base_model),
])
best_pipe.set_output(transform="pandas")
best_pipe.fit(X_train, y_train)

# evaluar en X_test crudo (hold‑out)
from sklearn.metrics import mean_squared_error

y_pred_test = best_pipe.predict(X_test)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
print("RMSE test:", rmse_test)


Mejor modelo básico: CatBoostRegressor
RMSE test: 259.08415318180636


### 4.3 Optimización (up to you 🫰🏻)

#### Ajuste nº1 - Lo subi 5to - 283.38416

In [22]:
cat_base = CatBoostRegressor(
    random_state=42,
    loss_function="RMSE",
    verbose=False,
)

pipe_cat = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", cat_base),
])

pipe_cat.set_output(transform="pandas")


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('screen',
                                                  FunctionTransformer(func=<function wrap_screen at 0x0000014A064EE2A0>),
                                                  ['ScreenResolution']),
                                                 ('cpu',
                                                  FunctionTransformer(func=<function wrap_cpu at 0x0000014A064EDA80>),
                                                  ['Cpu']),
                                                 ('memory',
                                                  FunctionTransformer(func=<function wrap_memory at 0x0000014A064EE7A0>),
                                                  ['Memory']),
                                                 ('gpu',
                                                  FunctionTr...
                                                  FunctionTransformer(func=<function wrap_weight at 0x0000014A064EE8E0>),
                                                  ['Weight']),
                                                 ('ram',
                                                  FunctionTransformer(func=<function wrap_ram at 0x0000014A064EE840>),
                                                  ['Ram']),
                                                 ('cat_ohe',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Company', 'TypeName',
                                                   'OpSys']),
                                                 ('inches', 'passthrough',
                                                  ['Inches'])],
                                   verbose_feature_names_out=False)),
                ('model',
                 <catboost.core.CatBoostRegressor object at 0x0000014A066671A0>)])

In [192]:
param_grid = {
    "model__depth": [4, 6, 8],
    "model__learning_rate": [0.03, 0.1],
    "model__iterations": [300, 600],
    "model__l2_leaf_reg": [1, 3, 5],
}

grid = GridSearchCV(
    estimator=pipe_cat,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)  # X_train crudo, el pipeline ya preprocesa dentro


Fitting 5 folds for each of 36 candidates, totalling 180 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('screen',
                                                                         FunctionTransformer(func=<function wrap_screen at 0x00000165F2201760>),
                                                                         ['ScreenResolution']),
                                                                        ('cpu',
                                                                         FunctionTransformer(func=<function wrap_cpu at 0x00000165F2202C00>),
                                                                         ['Cpu']),
                                                                        ('memory',
                                                                         FunctionTransformer(func=<function wrap_memory at 0x00000165F2202CA0>),
                                                                         [...
                                                                         ['Company',
                                                                          'TypeName',
                                                                          'OpSys']),
                                                                        ('inches',
                                                                         'passthrough',
                                                                         ['Inches'])],
                                                          verbose_feature_names_out=False)),
                                       ('model',
                                        <catboost.core.CatBoostRegressor object at 0x00000165972DA270>)]),
             n_jobs=-1,
             param_grid={'model__depth': [4, 6, 8],
                         'model__iterations': [300, 600],
                         'model__l2_leaf_reg': [1, 3, 5],
                         'model__learning_rate': [0.03, 0.1]},
             scoring='neg_mean_squared_error', verbose=1)

In [61]:
best_mse = -grid.best_score_
best_rmse = np.sqrt(best_mse)

print("Mejores hiperparámetros:", grid.best_params_)
print("Mejor MSE CV:", best_mse)
print("Mejor RMSE CV:", best_rmse)


Mejores hiperparámetros: {'model__depth': 4, 'model__iterations': 600, 'model__l2_leaf_reg': 1, 'model__learning_rate': 0.1}
Mejor MSE CV: 64946.90199387921
Mejor RMSE CV: 254.84682064699024


In [193]:
best_pipe_cat = grid.best_estimator_   # pipeline preprocess + CatBoost

y_pred_test = best_pipe_cat.predict(X_test)   # X_test crudo
rmse_test = root_mean_squared_error(y_test, y_pred_test)

print("RMSE test:", rmse_test)

RMSE test: 264.0091894334577


##### Veamos

In [194]:
cat_ajustado = CatBoostRegressor(
    random_state=42,
    loss_function="RMSE",
    verbose=False,
    depth =4,
    iterations= 600,
    l2_leaf_reg= 1,
    learning_rate= 0.1
)

In [195]:
cat_ajustado.fit(X_train_clean,y_train)

In [196]:
y_pred_test = cat_ajustado.predict(X_test_clean)
rmse_test = root_mean_squared_error(y_test, y_pred_test)
print("RMSE test:", rmse_test)

RMSE test: 277.90779422609904


#### Ajuste Nº 2 - Lo subi 6to - 278.60849

In [197]:
cat_base2 = CatBoostRegressor(
    random_state=42,
    loss_function="RMSE",
    verbose=False,
    depth = 4,
    learning_rate =  0.1 
    
)

pipe_cat2 = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", cat_base2),
])
pipe_cat2.set_output(transform="pandas")

param_grid2 = { 
    "model__iterations": [600, 800, 1000],
    "model__l2_leaf_reg": [0.5, 1, 2],
    "model__subsample": [0.7, 0.9, 1.0],
    "model__colsample_bylevel": [0.7, 0.9, 1.0],
}

grid2 = GridSearchCV(
    estimator=pipe_cat2,
    param_grid=param_grid2,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

grid2.fit(X_train, y_train)

best_mse2 = -grid2.best_score_
best_rmse2 = np.sqrt(best_mse2)
print("Mejores hiperparámetros 2ª búsqueda:", grid2.best_params_)
print("Mejor MSE CV 2ª:", best_mse2)
print("Mejor RMSE CV 2ª:", best_rmse2)


Fitting 5 folds for each of 81 candidates, totalling 405 fits
Mejores hiperparámetros 2ª búsqueda: {'model__colsample_bylevel': 0.9, 'model__iterations': 1000, 'model__l2_leaf_reg': 1, 'model__subsample': 0.9}
Mejor MSE CV 2ª: 63569.366092446275
Mejor RMSE CV 2ª: 252.12966127063726


In [198]:
best_pipe_cat2 = grid2.best_estimator_
y_pred_test2 = best_pipe_cat2.predict(X_test)
rmse_test2 = root_mean_squared_error(y_test, y_pred_test2)
print("RMSE test 2ª búsqueda:", rmse_test2)

RMSE test 2ª búsqueda: 272.5209043651393


#### Mismos hiperparametros que el modelo sin pipeline

In [211]:
model = CatBoostRegressor(loss_function="RMSE",
                          random_seed=42,
                          verbose=0,
                          depth =  4,
                          iterations = 750,
                          learning_rate= 0.1,
                          l2_leaf_reg= 1, 
                          min_data_in_leaf= 5, 
                          rsm= 0.8,
                          border_count= 64,
                          leaf_estimation_iterations= 5,
                          random_strength= 5,
                          subsample= 1.0)
pipe_manual = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", model),])

In [212]:
pipe_manual.set_output(transform="pandas")
pipe_manual.fit(X_train, y_train)
y_pred = pipe_manual.predict(X_test)
rmse_manual = root_mean_squared_error(y_test, y_pred)
print(f"RMSE Manual: {rmse_manual:.4f}")

RMSE Manual: 263.7930


In [213]:
rmse_cv = -cross_val_score(
    pipe_manual,
    X_train,
    y_train,
    cv=5,
    scoring="neg_root_mean_squared_error"
).mean()

rmse_cv

np.float64(256.0716954262874)

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [114]:
X_pred = pd.read_csv("./data/test.csv", index_col = 'laptop_ID')
X_pred = X_pred.drop(columns=["Product"])
X_pred.sample()

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,
800,Vero,Notebook,14.0,1920x1080,Intel Celeron Dual Core N3350 1.1GHz,4GB,32GB Flash Storage,Intel HD Graphics 500,Windows 10,1.22kg


In [115]:
# X_pred.tail()

In [116]:
# X_pred.info()

In [117]:
preprocessor.set_output(transform="pandas")

X_pred_clean = preprocessor.fit_transform(X_pred)
X_pred_clean.head()

,Res_X,Res_Y,Touchscreen,Screen_Qual_Ord,CPU_Intel,CPU_AMD,CPU_Other,CPU_Core_Ord,CPU_CoreM_Ord,CPU_Ryzen_Ord,...,TypeName_Workstation,OpSys_Chrome OS,OpSys_Linux,OpSys_Mac OS X,OpSys_No OS,OpSys_Windows 10,OpSys_Windows 10 S,OpSys_Windows 7,OpSys_macOS,Inches
laptop_ID,,,,,,,,,,,,,,,,,,,,,
209,1920,1080,0,2,1,0,0,7,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,15.6
1281,1366,768,0,0,1,0,0,0,0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,15.6
1168,1366,768,0,0,1,0,0,3,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,15.6
1231,1920,1080,1,2,1,0,0,5,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,15.6
1020,1920,1080,0,2,1,0,0,5,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,14.0


In [118]:
X_pred_clean.shape

(391, 76)

In [121]:
X_pred_clean.columns

Index(['Res_X', 'Res_Y', 'Touchscreen', 'Screen_Qual_Ord', 'CPU_Intel',
       'CPU_AMD', 'CPU_Other', 'CPU_Core_Ord', 'CPU_CoreM_Ord',
       'CPU_Ryzen_Ord', 'CPU_ASeries_Ord', 'CPU_Celeron', 'CPU_Pentium',
       'CPU_Atom', 'CPU_ESeries', 'CPU_OtherFamily', 'CPU_GHz', 'SSD_GB',
       'HDD_GB', 'Flash_Storage_GB', 'Hybrid_GB', 'GPU_Intel', 'GPU_Nvidia',
       'GPU_AMD', 'GPU_Other', 'GPU_Dedicada', 'GPU_GTX', 'GPU_RTX', 'GPU_MX',
       'GPU_Quadro', 'GPU_RX', 'GPU_FirePro', 'GPU_GTX_Num', 'GPU_MX_Num',
       'GPU_Quadro_Num', 'GPU_RX_Num', 'GPU_Iris', 'GPU_HD', 'GPU_UHD',
       'GPU_Intel_Num', 'GPU_Radeon_Num', 'Weight_kg', 'Ram_GB',
       'Company_Acer', 'Company_Apple', 'Company_Asus', 'Company_Chuwi',
       'Company_Dell', 'Company_Fujitsu', 'Company_Google', 'Company_HP',
       'Company_LG', 'Company_Lenovo', 'Company_MSI', 'Company_Mediacom',
       'Company_Microsoft', 'Company_Razer', 'Company_Samsung',
       'Company_Toshiba', 'Company_Vero', 'Company_Xiaomi',
    

 ## 2. Replicar el procesado para ``test.csv``

In [119]:
# X_pred_clean

In [123]:
predictions_submit = best_pipe_cat2.predict(X_pred)
predictions_submit

array([1531.03909085,  271.69765203,  335.88559577, 1010.52059944,
        955.84210007,  500.79418911,  821.07438622,  872.64117044,
       1248.69574997,  250.43464623, 2155.78309608, 1420.38889985,
        495.03643721, 1560.92537959,  815.8300536 ,  780.04184732,
       2373.13851539, 1290.1480464 , 1985.50878459,  708.19530866,
       1545.37057829,  263.20007761,  844.0629122 , 1182.8845244 ,
        378.99468279,  755.36087878,  597.58791859,  937.07539691,
       3378.94417835, 1093.51837151, 2418.9034456 ,  425.8227862 ,
        840.50426751, 2861.42149601, 2258.47633236, 1580.42674777,
        631.52443036, 1347.19549909,  989.57446542, 1756.25789419,
        672.20599865,  718.33714477,  543.07997744, 1272.88981768,
       1115.29405057, 1053.6265232 , 1037.6570435 ,  631.45718123,
        672.97517722,  398.02412839, 1629.84428228,  782.56439299,
       1133.24620678,  322.02802429, 1934.57105315, 1755.70110774,
        759.15358138,  960.78938235,  988.73281755,  611.07163

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [124]:
# ¿Qué opináis?
# ¿Sí, no?

![wow.jpeg](attachment:wow.jpeg)

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [125]:
sample = pd.read_csv("data/sample_submission.csv")

In [126]:
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


In [127]:
sample.shape

(391, 2)

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [128]:
#¿Cómo creamos la submission?
submission = pd.DataFrame({
    'laptop_ID': X_pred.index,
    'Price_in_euros' : predictions_submit
})

In [129]:
submission.head()

,laptop_ID,Price_in_euros
0,209,1531.039091
1,1281,271.697652
2,1168,335.885596
3,1231,1010.520599
4,1020,955.842100


In [100]:
submission.shape

(391, 2)

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [130]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission_6.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [131]:
chequeador(submission)

You're ready to submit!
